# 04 · Solvers: QAOA on Qiskit Aer and simulated annealing

**What it does.** Runs the two solvers of the project on the Stage 1 QUBO:
`run_qaoa` (a hand-written QAOA loop on the Qiskit Aer simulator) and `run_sa` (simulated annealing from
D-Wave's `dwave-samplers`, the maintained successor of `neal`). `distribution_stats` turns their output into
the reported metrics.

**Why it was implemented.** TQ2 asks how QAOA or annealing can be implemented. The QAOA loop is written out
rather than taken from a library wrapper so every step can be explained in the report.

**Quantum terms in one place.**
- *Qubit*: a bit that, until read, holds a weighted mix (*superposition*) of 0 and 1; 8 qubits hold weights for all 256 bitstrings.
- *Measurement / shots*: reading the qubits gives one bitstring at random according to those weights; 4096 *shots* = a sample of 4096.
- *Ising Hamiltonian*: the QUBO rewritten with ±1 spins ($x = (1 - z)/2$) as an operator $H$ of Z and ZZ terms; its value on a bitstring is that bitstring's energy.
- *QAOA*: a small parameterised model. Start with all bitstrings equally likely, then repeat $p$ *layers* of a cost step $e^{-i\gamma H_C}$ (phase by energy) and a *mixer* step $e^{-i\beta \sum X}$ (moves weight between bitstrings). The circuit template is the *ansatz*; a classical optimiser trains the $2p$ angles $(\gamma, \beta)$ to minimise expected energy, like model weights.
- *Transpile*: compile the circuit into the gates a simulator or device supports.
- *Simulated annealing*: a classical algorithm; random bit flips, worse moves accepted less often as the temperature falls. The project reports it as "neal / simulated annealing".

**Resources.**
- Code: `common/solvers.py` (`run_qaoa`, `run_sa`, `HARDWARE_BASIS`), `common/results.py` (`distribution_stats`), settings in `configs/vrp.yaml`.
- Libraries: Qiskit 2.6 (`qaoa_ansatz`, `transpile`), Qiskit Aer 0.17 (`EstimatorV2`, `SamplerV2`), qiskit-optimization 0.7 (`to_ising`), SciPy 1.18 (`minimize`, Powell), dwave-samplers 1.8.
- Course: hybrid loop and QAOA, `Quantum_Lecture 2` slides 9 and 16; shots, `Lecture 3` slide 14.
- Literature: E. Farhi, J. Goldstone, S. Gutmann, *A Quantum Approximate Optimization Algorithm*, arXiv:1411.4028 (2014); S. Kirkpatrick et al., *Optimization by Simulated Annealing*, Science 1983; M. J. D. Powell, Computer Journal 1964.

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
import yaml
from common.qubo_utils import verify_formulation
from domains.vrp.problem import build_bqm, build_qubo, decode, is_feasible, load_instance

cfg = yaml.safe_load(Path("configs/vrp.yaml").read_text(encoding="utf-8"))
inst = load_instance(cfg["instance"], cfg["n_vehicles"])
penalties = cfg["stage1"]["penalties"]
bqm, qp = build_bqm(inst, penalties), build_qubo(inst, penalties)
feasible = lambda s: is_feasible(decode(s, inst))
check = verify_formulation(bqm, feasible)
print("Stage 1 QUBO:", bqm.num_variables, "variables | brute-force ground energy:", check["ground_energy"])

## Step 1–2: QUBO → Ising, then normalise
Qubit $i$ is QUBO variable $i$ (`qp.variables`). Pauli labels are little-endian: the rightmost character is
qubit 0. `run_qaoa` divides the operator by its largest coefficient $s$ so the angle $\gamma$ is of order 1
regardless of the penalty size (the reason is shown in notebook 07).

In [ ]:
ising, offset = qp.to_ising()
scale = max(abs(c) for c in ising.coeffs)
print("qubit -> variable:", {i: v.name for i, v in enumerate(qp.variables)})
print("Pauli terms:", len(ising), "| constant offset:", offset, "| largest coefficient s:", scale)
for label, coeff in sorted(ising.to_list(), key=lambda t: -abs(t[1]))[:5]:
    print(f"  {label}  {coeff.real:+.2f}")

## Step 3: the ansatz and its cost in gates
Qiskit orders circuit parameters alphabetically (all β before all γ), so `run_qaoa` binds them by name.
Depth is reported after transpiling to a generic hardware gate set (CNOT, RZ, SX, X): each layer needs
2 CNOTs per QUBO coupling.

In [ ]:
from qiskit import transpile
from qiskit.circuit.library import qaoa_ansatz
from common.solvers import HARDWARE_BASIS

print("parameters, p = 2:", [p.name for p in qaoa_ansatz(ising * (1 / scale), reps=2).parameters])
depth = {}
for p in (1, 2, 3, 4, 5):
    hw = transpile(qaoa_ansatz(ising * (1 / scale), reps=p), basis_gates=HARDWARE_BASIS, seed_transpiler=1)
    depth[p] = (hw.depth(), hw.count_ops().get("cx", 0))
    print(f"p = {p}: depth {depth[p][0]:>3}, CNOTs {depth[p][1]:>3}")

## Step 4–5: train the angles and sample (the reported configuration)
p = 4, Powell, 300 energy evaluations, initial γ = 1 and β = 0.4, 4096 shots, seeds 1 (`configs/vrp.yaml`).
Aer's `EstimatorV2` returns the exact expected energy during training; `SamplerV2` draws the final shots.
About 5–30 seconds depending on the machine.

In [ ]:
from common.results import distribution_stats
from common.solvers import run_qaoa

q = cfg["qaoa"]
qaoa = run_qaoa(qp, reps=q["reps"], initial_gammas=[q["initial_gamma"]] * q["reps"],
                initial_betas=[q["initial_beta"]] * q["reps"], optimizer=q["optimizer"], maxiter=q["maxiter"],
                shots=q["shots"], seed_transpiler=q["seed_transpiler"], seed_simulator=q["seed_simulator"])
qaoa_stats = distribution_stats(qaoa["distribution"], feasible, check["ground_energy"])
print(f"energy evaluations: {qaoa['n_evaluations']} | expected energy after training: {qaoa['expected_energy']:.1f}")
print(f"p_optimal = {qaoa_stats['p_optimal']:.3f} | p_feasible = {qaoa_stats['p_feasible']:.3f} "
      f"| uniform sampling would give {1 / 2 ** bqm.num_variables:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(qaoa["energy_history"], color="#2a78d6", linewidth=1.5)
ax.axhline(check["ground_energy"], color="#52514e", linestyle="--", linewidth=1, label="optimal energy (134)")
ax.set_xlabel("energy evaluation")
ax.set_ylabel("expected QUBO energy")
ax.set_title("QAOA training (Powell, p = 4)")
ax.legend()
plt.show()

print("most frequent measured bitstrings:")
for r in sorted(qaoa["distribution"], key=lambda r: -r["count"])[:8]:
    print(f"  {decode(r['sample'], inst).routes}  P = {r['count'] / q['shots']:.3f}  energy {r['energy']:g}"
          f"  feasible {feasible(r['sample'])}")

## Simulated annealing on the same QUBO (1000 reads × 1000 sweeps, seed 1)

In [ ]:
from common.solvers import run_sa

s = cfg["sa"]
sa = run_sa(bqm, num_reads=s["num_reads"], num_sweeps=s["num_sweeps"], seed=s["seed"])
sa_stats = distribution_stats(sa["distribution"], feasible, check["ground_energy"])
print(f"p_optimal = {sa_stats['p_optimal']:.3f} | p_feasible = {sa_stats['p_feasible']:.3f} | runtime {sa['runtime_s']:.2f} s")

## Expected results
- 8 qubits; Ising offset 871.5; largest coefficient s = 134.25.
- Parameters ordered β[0], β[1], γ[0], γ[1].
- Hardware-basis CNOTs: 32 per layer (32, 64, 96, 128, 160); depth 39, 60, 81 for p = 1–3.
- QAOA (p = 4, Powell): p_optimal ≈ **0.052**, p_feasible ≈ **0.296**, about 13× uniform sampling (0.0039), yet ~70% of shots infeasible.
- Simulated annealing: p_optimal ≈ **0.507**, p_feasible ≈ **0.992**.

Seeds are fixed, so with the pinned versions the numbers reproduce exactly; the checks allow small tolerance.

In [ ]:
assert bqm.num_variables == 8 and offset == 871.5 and abs(scale - 134.25) < 1e-9
assert [p for p in depth if depth[p][1] != 32 * p] == []
assert abs(qaoa_stats["p_optimal"] - 0.052) < 0.005 and abs(qaoa_stats["p_feasible"] - 0.296) < 0.02
assert abs(sa_stats["p_optimal"] - 0.507) < 0.02 and sa_stats["p_feasible"] > 0.98